# Week 3 · 训练工程化

> **本周一句话**:模型结构一字不改,给训练循环加 4 样东西(LR 调度 / AMP / 梯度裁剪 / weight decay),让训练从"能跑"变成"健康"。

上周 v0.3 用恒定 `lr=3e-4` 莽训能到 val 4.14,本周把训练循环升级到 GPT-2 同款配置 —— val 数字不会显著降(因为模型结构没变),但训练稳定性、显存占用、对超参的容忍度全面提升,**为 Week 5 25M 模型打基础**。

## 0. 本周目标

| 版本 | 新增 | 参数量 | val loss | 训练速度 | 备注 |
|---|---|---|---|---|---|
| v0.3(上周) | — | 6.37M | 4.14 | 1× | 恒定 lr,fp32 |
| v0.4 | + warmup + cosine LR | 6.37M | 4.21~4.24 | 1× | 训练曲线更平滑 |
| v0.5 | + AMP + grad clip + wd | 6.37M | 类似 v0.4 | **1.8×** | 显存省一半 |

看数字像没提升 —— 但你升了 4 项工程化能力,**Week 5 没有它们就训不动 25M**。

## 1. 前置知识

**必备**:
- Week 2 跑完(v0.3 val 4.14 见过)
- 知道什么是"梯度爆炸 / 消失"

**这周第一次遇到**:
- 学习率调度器 (`get_lr(step)`)
- 自动混合精度 (`torch.amp.autocast` + `GradScaler`)
- 梯度裁剪 (`clip_grad_norm_`)
- AdamW 的 `weight_decay` 和 `betas=(0.9, 0.95)`

## 2. 核心概念

### 2.1 Warmup + Cosine Decay

恒定 lr 的问题:训练初期权重还是随机的,大 lr 会把权重打飞;训练后期接近收敛,大 lr 又会让模型在最优点附近震荡。

**正确做法**:两阶段调度。

```
 lr
  ↑
  │     ╱──╲___              ← cosine decay 平滑降到 min_lr
  │    ╱     ╲___
  │   ╱          ╲___
  │  ╱               ╲___
  │ ╱                    ╲___
  │╱                          ╲
  ├──┼──────────────────────────→ step
  0 warmup_steps           total
     (200)               (8000)
```

- **阶段 1 warmup**(前 200 步):lr 从 0 线性升到 max_lr=3e-4
- **阶段 2 cosine**(其余):按 cosine 曲线平滑降到 min_lr=3e-5

warmup 让模型"暖身",cosine 让后期"收手"。GPT-2/GPT-3/Llama 都这么做。

In [ ]:
# 完整调度函数(对应 train/lr_schedule.py)
import math

def warmup_cosine(step, warmup, total, max_lr, min_lr):
    if step < warmup:
        return max_lr * (step + 1) / warmup
    progress = (step - warmup) / max(total - warmup, 1)
    progress = min(progress, 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr + coeff * (max_lr - min_lr)

# 打印几个关键点确认形状
warmup, total = 200, 8000
mx, mn = 3e-4, 3e-5
for s in [0, 100, 200, 500, 2000, 4000, 6000, 7999]:
    print(f"  step {s:5d}: lr = {warmup_cosine(s, warmup, total, mx, mn):.6f}")

### 2.2 AMP(自动混合精度)

默认 PyTorch 用 fp32(32 位浮点)。AMP 让 forward / backward 用 **fp16(16 位)**,权重和优化器状态仍用 fp32 保留精度。

**好处**:
- 显存占用约减半(activation 是显存大头,fp16 占一半)
- T4 / A100 上 fp16 矩阵乘 1.5-3× 快
- **几乎不损失精度**(关键操作仍是 fp32)

**两个工具**:

```python
with autocast(device_type="cuda", dtype=torch.float16):
    logits, loss = model(x, y)        # forward 在 fp16

scaler.scale(loss).backward()          # 反向前把 loss 放大,防止小梯度变 0
scaler.unscale_(optimizer)             # 真正更新前缩回正常尺度
torch.nn.utils.clip_grad_norm_(...)    # ← 必须在 unscale_ 之后!
scaler.step(optimizer)                 # 把 fp16 梯度合并到 fp32 主权重
scaler.update()                        # 动态调整下次的 scale 大小
```

**核心机制**:fp16 表示范围小,梯度数值经常下溢成 0。`GradScaler` 先把 loss 乘一个大数(scale)放大梯度,反向后再缩回去 —— 让原本会变 0 的小梯度保留下来。

### 2.3 梯度裁剪(Grad Clip)

深层 Transformer 偶尔会出现单步梯度爆炸(比如某个 batch 数值特别极端),一步把训练打飞,后面 loss 再也下不来。

**修复**(一行):

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

计算所有梯度的 L2 范数,如果超过 1.0 就按比例缩回 1.0。**梯度方向不变,只是步长被砍到安全范围**。

`max_norm=1.0` 是 LLM 训练的标准值,GPT-2/3/Llama 都这么用。

**和 AMP 配合的顺序很重要**:

```python
scaler.scale(loss).backward()      # 梯度此时还是被 scale 放大过的
scaler.unscale_(optimizer)         # 缩回正常尺度
clip_grad_norm_(...)               # 这里 clip 才有意义
scaler.step(optimizer)
```

顺序错了 → clip 在 scale 后的尺度上做,等于没 clip。

### 2.4 AdamW 的 weight_decay 和 betas

上周 v0.3 我们直接用 `torch.optim.AdamW(model.parameters(), lr=3e-4)`,默认 `weight_decay=0.01`、`betas=(0.9, 0.999)`。

本周改成 GPT 同款:

```python
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=max_lr,
    weight_decay=0.1,        # ← 从 0.01 调到 0.1,L2 惩罚更强
    betas=(0.9, 0.95),       # ← 第二个 beta 从 0.999 → 0.95
)
```

**weight_decay=0.1 的逻辑**:在每步更新时,把权重往 0 拉一点(`w ← w - lr * 0.1 * w`)。强一点的 wd 防过拟合,LLM 上 0.1 是经验值。

**betas=(0.9, 0.95) 的逻辑**:`betas[1]` 控制对历史梯度二阶矩(方差)的平滑。默认 0.999 在大 batch / 长训练时太平滑,模型对最近梯度反应慢。GPT-2 / Llama 都用 0.95,反应快一点。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `train/lr_schedule.py` | 30 | `warmup_cosine(step, ...)` + `apply_lr(optimizer, lr)` |
| `train/train_v04.py` | 75 | v0.3 结构 + LR schedule |
| `train/train_v05.py` | 95 | v0.4 + AMP + grad clip + weight_decay |
| `configs/config.py:TrainV05Cfg` | 8 | 8000 / 200 / 3e-4 / 3e-5 / 0.1 / 1.0 |

v04 → v05 的 diff 主要在 train 循环中间几行;模型类是同一个。

## 4. 动手做

In [ ]:
import subprocess, sys
from pathlib import Path


def _find_repo_root() -> Path:
    """定位仓库根目录(含 pyproject.toml + train/),不依赖 notebook 工作目录。
    本地/Colab 的 cwd 在 courses/,往上一级命中;
    魔搭 ModelScope 的 cwd 在工作区根(= 仓库根),当场命中。"""
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "pyproject.toml").is_file() and (d / "train").is_dir():
            return d
    # 兜底:往下找两层(仓库被克隆进子目录的情况)
    for f in (*here.glob("*/pyproject.toml"), *here.glob("*/*/pyproject.toml")):
        if (f.parent / "train").is_dir():
            return f.parent.resolve()
    raise RuntimeError(f"找不到仓库根(应含 pyproject.toml + train/),当前 cwd={here}")


REPO = _find_repo_root()                      # 项目根的绝对路径
print(f"REPO = {REPO}")


def run(cmd):
    """跑子进程并把输出实时打印到 cell。
    - "python" 换成 sys.executable,确保用当前 kernel 解释器。
    - 形如 "../train/x.py" 的相对路径统一解析成 REPO 下的绝对路径,
      不再依赖 notebook 的工作目录(魔搭与本地/Colab 的 cwd 不一致)。
    - cwd=REPO 兜底:即便脚本内部用了相对路径也能找到文件。
    - Popen 逐行回读才能在 cell 里实时看到脚本的 print。"""
    cmd = [sys.executable if c == "python" else c for c in cmd]
    cmd = [str(REPO / c.removeprefix("../")) if isinstance(c, str) and c.startswith("../") else c
           for c in cmd]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1, cwd=str(REPO),
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"{cmd} 退出码 {proc.returncode}")

# 先看一下 lr schedule 是什么形状
run(["python", "../train/lr_schedule.py"])

In [ ]:
# v0.4: 加 LR schedule
run(["python", "../train/train_v04.py"])
# 预期: val 4.21 ~ 4.24(数字未必比 v0.3 低,但训练曲线更平滑)
# ~8 分钟(T4)

In [ ]:
# v0.5: 加 AMP + grad clip + weight decay
run(["python", "../train/train_v05.py"])
# 预期: val 接近 v0.4,但训练快 ~1.8× ,显存少 ~40%
# ~6 分钟(T4)

**看什么数字判断成功**:
- v0.4 的 lr 字段每步都不一样(warmup 阶段从 0 线性涨),v0.3 一直 3e-4
- v0.5 比 v0.4 实际墙钟时间快(同样 8000 步,少 1-2 分钟)
- v0.5 训练时 `nvidia-smi` 看显存,比 v0.4 低

## 5. 自测题

**A. LR Schedule**
- A1 不加 warmup 直接从 max_lr=3e-4 开始会怎样?用 v0.3 实验过的数据告诉你答案
- A2 cosine decay 把 lr 降到 min_lr 而不是 0,为什么不直接降到 0?
- A3 8000 步训练,warmup 200 步占总训练的 2.5%,这个比例是怎么定的?换成 1000 步会有什么影响?

**B. AMP**
- B1 如果某个梯度的真实值是 1e-8,fp16 能表示吗?GradScaler 怎么帮它存活?
- B2 forward 用 fp16、权重用 fp32 —— 这两份权重是什么时候同步的?
- B3 T4 / A100 / 消费级 RTX 哪些 GPU 对 fp16 矩阵乘有硬件加速?为什么 V100 之前的卡用 AMP 收益小?

**C. Grad Clip**
- C1 max_norm=1.0 太严会怎样?太松(比如 100)会怎样?
- C2 为什么用 L2 范数而不是按每个参数单独裁剪?(也就是 clip_grad_value vs clip_grad_norm)
- C3 如果某一步 clip 把梯度缩了 100×,要不要担心?要看哪个指标判断这是常态还是异常?

**D. Weight Decay**
- D1 AdamW 的 W = decoupled weight decay,和"在 loss 里加 0.1 * ||w||^2"等价吗?差在哪?
- D2 weight_decay 加到 Linear 的 bias 和 LayerNorm 的 weight 上合理吗?(GPT-2 实际不加这些)
- D3 betas=(0.9, 0.999) vs (0.9, 0.95) 在我们 6.37M 模型上能看出明显差别吗?在 1B 模型上呢?

## 6. 容易踩的坑

**坑 1:`clip_grad_norm_` 写在 `scaler.unscale_` 之前**

梯度还在 scale 后的尺度上,clip 等于没 clip,但 `step` 时又会自动 unscale —— 实际相当于完全没 clip。**症状**:训练偶尔崩飞,loss 突然变 nan。

**坑 2:`autocast` 套到 `estimate_loss` 外面**

评估时也要用 fp16(否则 train/val 不同精度对比有偏差)。我们的 `estimate_loss` 内部已经套了 autocast。

**坑 3:cosine decay 的 total_steps 算错**

resume 训练时如果 total_steps 没相应延长,模型会在 lr 已经降到 min_lr 之后还硬训 —— 学不动新东西。Week 4 的 resume 会用一个新的小 lr 重启 schedule(不是接着上次的 schedule)。

**坑 4:AMP 在 CPU 上跑会报错或退化到 fp32**

`autocast(device_type="cuda", ...)` 显式指定 device_type。如果你在 CPU 上跑这段代码,要么改成 `device_type="cpu"` + bfloat16,要么直接跳过 AMP。

**坑 5:`GradScaler` 第一步 `inf check` 失败 → 静默不更新**

GradScaler 会探测梯度是否溢出 → 溢出就跳过这一步并降低 scale。第一次启动 scale 偏高时可能跳过几步,看着像"loss 不降",其实是正常 warmup。**别看到第一步没更新就慌**。

## 7. 进入 Week 4 前

现在你应该:
- ☑ 跑过 train_v04 和 train_v05,看到 lr 不再是常数
- ☑ 能解释 GradScaler 为什么必须存在(不是装饰)
- ☑ 知道 `clip_grad_norm_` 必须在 `unscale_` 之后
- ☑ 体感到 v0.5 比 v0.4 快、占显存少

Week 4 我们做完整训练工程闭环:checkpoint 系统 + 断点续训 + TensorBoard 监控。把"实验室代码"升级成"工业代码"。